## ML Algorithms

In [1]:
import numpy as np

### Multi-Head Attention

In [2]:
class MultiHeadAttention:
    def __init__(self, embed_dim, n_heads):
        self.head_dim = embed_dim // n_heads
        self.embed_dim = embed_dim
        self.n_heads = n_heads

        self.W_q = np.random.rand(embed_dim, embed_dim)
        self.W_k = np.random.rand(embed_dim, embed_dim)
        self.W_v = np.random.rand(embed_dim, embed_dim)
        self.W_o = np.random.rand(embed_dim, embed_dim)

    def network(self, x):
        q = x @ self.W_q
        k = x @ self.W_k
        v = x @ self.W_v
        return q, k, v

    def forward(self, x, mask=None):
        B, seq_length, embed_dim = x.shape
        
        q, k, v = self.network(x)
        q = q.reshape(B, seq_length, self.n_heads, self.head_dim).swapaxes(1, 2)
        k = k.reshape(B, seq_length, self.n_heads, self.head_dim).swapaxes(1, 2)
        v = v.reshape(B, seq_length, self.n_heads, self.head_dim).swapaxes(1, 2)

        weight = (q @ k.swapaxes(-1, -2)) / np.sqrt(self.head_dim)
        if mask is not None:
            weight += (mask * -1e9)
        weight = np.exp(weight - np.max(weight, axis=-1, keepdims=True))
        weight /= np.sum(weight, axis=-1, keepdims=True)
        
        out = weight @ v
        out = out.swapaxes(1, 2).reshape(B, seq_length, embed_dim)

        return out @ self.W_o

In [3]:
x = np.random.rand(32, 100, 256)
mha = MultiHeadAttention(256, 8)
mha.forward(x).shape

(32, 100, 256)

### Multi-Head Attention (LoRA)

In [4]:
class MultiHeadAttention:
    def __init__(self, embed_dim, n_heads, rank, alpha):
        self.head_dim = embed_dim // n_heads
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.rank = rank
        self.alpha = alpha

        self.W_q = np.random.rand(embed_dim, embed_dim)
        self.W_k = np.random.rand(embed_dim, embed_dim)
        self.W_v = np.random.rand(embed_dim, embed_dim)
        self.W_o = np.random.rand(embed_dim, embed_dim)

        self.Aq = np.random.rand(rank, embed_dim)
        self.Bq = np.zeros((embed_dim, rank))

        self.Ak = np.random.rand(rank, embed_dim)
        self.Bk = np.zeros((embed_dim, rank))

        self.Av = np.random.rand(rank, embed_dim)
        self.Bv = np.zeros((embed_dim, rank))

    def network(self, x):
        q = x @ self.W_q + (self.alpha / self.rank) * (x @ self.Bq) @ self.Aq
        k = x @ self.W_k + (self.alpha / self.rank) * (x @ self.Bk) @ self.Ak
        v = x @ self.W_v + (self.alpha / self.rank) * (x @ self.Bv) @ self.Av
        return q, k, v

    def forward(self, x, mask=None):
        B, seq_length, embed_dim = x.shape
        
        q, k, v = self.network(x)
        q = q.reshape(B, seq_length, self.n_heads, self.head_dim).swapaxes(1, 2)
        k = k.reshape(B, seq_length, self.n_heads, self.head_dim).swapaxes(1, 2)
        v = v.reshape(B, seq_length, self.n_heads, self.head_dim).swapaxes(1, 2)

        weight = (q @ k.swapaxes(-1, -2)) / np.sqrt(self.head_dim)
        if mask is not None:
            weight += (mask * -1e9)
        weight = np.exp(weight - np.max(weight, axis=-1, keepdims=True))
        weight /= np.sum(weight, axis=-1, keepdims=True)
        
        out = weight @ v
        out = out.swapaxes(1, 2).reshape(B, seq_length, embed_dim)

        return out @ self.W_o

In [5]:
x = np.random.rand(32, 100, 256)
mha = MultiHeadAttention(256, 8, 8, 1.0)
mha.forward(x).shape

(32, 100, 256)

### Notes

**Why `np.sqrt(self.head_dim)`?**

The score is `q · k = Σ qᵢkᵢ` over `d_k = head_dim` terms. If `q`, `k` have ~zero mean and unit variance, each product has variance ~1, so the sum has variance ~`d_k` and grows like `√d_k`. Dividing by `√d_k` rescales the logits back to unit variance.

**Why `√d_k` specifically?**

The standard deviation grows as `√d_k`, not `d_k`. Without scaling, large `d_k` produces large logits that push `softmax` into a saturated, near one-hot regime where gradients vanish. Dividing by `√d_k` keeps `softmax` well-conditioned with healthy gradients (from *Attention Is All You Need*).

**Complexity** 

(seq length `n`, model dim `d`, heads `h`, `d_k = d/h`)
- Q/K/V/output projections: `O(n·d²)`
- Scores `q @ kᵀ` and `weight @ v`: `O(n²·d)`
- **Time:** `O(n²·d + n·d²)` — **Memory:** `O(n² + n·d)`, dominated by the `n × n` attention matrix.

The `n²` term is the bottleneck for long sequences.

**Making it efficient for long sequences**

- **FlashAttention** — tiling + online softmax; never materializes the `n×n` matrix. Exact result, `O(n)` memory, IO-aware speedups.
- **Sparse attention** (Longformer, BigBird) — local window + a few global tokens → `O(n·w)`.
- **Linear attention** (Performer) — kernel feature map reorders `softmax(QKᵀ)V ≈ φ(Q)(φ(K)ᵀV)` → `O(n·d²)`, linear in `n`.
- **Low-rank** (Linformer) — project K/V from length `n → k` → `O(n·k)`.
- **MQA / GQA + KV caching** — share K/V across heads and reuse past K/V; shrinks the KV cache for autoregressive inference.


### IoU + NMS

In [6]:
def box_iou(boxes1, boxes2):
    """
    boxes: (x1, y1, x2, y2) format
    """
    # area = width x height
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])

    lt = np.maximum(boxes1[:, None, :2], boxes2[:, :2])  # (N, M, 2)
    rb = np.minimum(boxes1[:, None, 2:], boxes2[:, 2:])  # (N, M, 2)
    wh = np.clip(rb - lt, a_min=0, a_max=None)  # (N, M, 2)
    
    inter = wh[:, :, 0] * wh[:, :, 1]  # (N, M)
    union = area1[:, None] + area2 - inter  # (N, M)
    iou = inter / union  # (N, M)
    return iou

def nms(boxes, scores, iou_threshold):
    idxs = np.argsort(scores)[::-1]
    keep = []
    
    while len(idxs) > 0:
        current = idxs[0]
        keep.append(current)
        
        if len(idxs) == 1:
            break
        
        ious = box_iou(boxes[current:current+1], boxes[idxs[1:]])[0]
        idxs = idxs[1:][ious < iou_threshold]
    
    return keep

In [7]:
boxes1 = np.random.rand(5, 4)
boxes2 = np.random.rand(5, 4)

print(box_iou(boxes1, boxes2))
print(nms(boxes1, np.random.rand(5), 0.5))

[[ 0.  0. -0.  0. -0.]
 [ 0.  0. -0.  0. -0.]
 [-0. -0. -0. -0. -0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]]
[3, 2, 1, 4, 0]


### Billinear Interpolation (Image resize)

In [8]:
def bilinear_resize(img, new_h, new_w):
    C, H, W = img.shape
    ys = np.linspace(0, H - 1, new_h)
    xs = np.linspace(0, W - 1, new_w)

    y0 = np.clip(np.floor(ys).astype(int), 0, H - 2)   # (new_h,)
    x0 = np.clip(np.floor(xs).astype(int), 0, W - 2)   # (new_w,)
    y1, x1 = y0 + 1, x0 + 1

    wy = (ys - y0)[:, None]   # (new_h, 1)
    wx = (xs - x0)[None, :]   # (1, new_w)

    I00 = img[:, y0[:, None], x0[None, :]]
    I01 = img[:, y0[:, None], x1[None, :]]
    I10 = img[:, y1[:, None], x0[None, :]]
    I11 = img[:, y1[:, None], x1[None, :]]

    return ((1 - wy) * (1 - wx) * I00 + (1 - wy) * wx * I01 +
            wy * (1 - wx) * I10 + wy * wx * I11)

In [9]:
img = np.random.rand(3, 100, 100)
resized_img = bilinear_resize(img, 50, 50)
print(img.shape, resized_img.shape)

(3, 100, 100) (3, 50, 50)
